# Bronze to Silver Transformation
## Pipeline Monitor — Databricks Notebook 01

This notebook reads raw Parquet files from the Bronze zone,
cleanses and validates the data, and writes Delta Lake tables to Silver.

### Medallion Architecture
- **Bronze** → Raw data as-landed from APIs (immutable)
- **Silver** → Cleaned, typed, deduplicated (this notebook)
- **Gold** → Business-ready aggregations (notebook 02)

In [0]:
# Cell 1 — Configuration
# Unity Catalog compatible — no mounting needed
# Uses ABFS (Azure Blob File System) direct paths

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime

# Read secrets securely from Databricks Secret Scope
STORAGE_ACCOUNT = dbutils.secrets.get(scope="pipeline-monitor", key="adls-account-name")
STORAGE_KEY     = dbutils.secrets.get(scope="pipeline-monitor", key="adls-account-key")
TODAY           = datetime.utcnow().strftime("%Y/%m/%d")

# Configure Spark to access ADLS using the storage key
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    STORAGE_KEY
)

# Define base paths using ABFS protocol
BRONZE = f"wasbs://bronze@{STORAGE_ACCOUNT}.blob.core.windows.net"
SILVER = f"wasbs://silver@{STORAGE_ACCOUNT}.blob.core.windows.net"
GOLD   = f"wasbs://gold@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"✅ Storage configured: {STORAGE_ACCOUNT}")
print(f"✅ Today: {TODAY}")
print(f"✅ Bronze path: {BRONZE}")
print(f"✅ Silver path: {SILVER}")
print(f"✅ Gold path: {GOLD}")

## Step 1: Orders Bronze → Silver

Reads raw order Parquet files, fixes data types,
removes duplicates, handles nulls, and writes
clean Delta Lake table to Silver zone.

Key transformations:
- Cast order_id, customer_id, product_id to Long
- Parse order_timestamp to proper timestamp
- Cast amount to Decimal(18,2)
- Uppercase currency, lowercase status
- Deduplicate by order_id (keep latest)

In [0]:
# Cell 2 — Orders: Bronze → Silver
print("\n[1/4] Processing orders Bronze → Silver...")

# Read raw Parquet from Bronze
orders_raw = spark.read.parquet(f"{BRONZE}/orders/{TODAY}/")
print(f"  Raw rows: {orders_raw.count()}")
print(f"  Raw columns: {orders_raw.columns}")

# Define deduplication window
dedup_window = Window.partitionBy("order_id").orderBy(F.col("ingested_at").desc())

# Cleanse and validate
orders_silver = (
    orders_raw
    .withColumn("order_id",        F.col("order_id").cast(LongType()))
    .withColumn("customer_id",     F.col("customer_id").cast(LongType()))
    .withColumn("product_id",      F.col("product_id").cast(LongType()))
    .withColumn("order_timestamp", F.to_timestamp("order_timestamp"))
    .withColumn("amount",          F.col("amount").cast(DecimalType(18,2)))
    .withColumn("currency",        F.upper(F.trim("currency")))
    .withColumn("status",          F.lower(F.trim("status")))
    .withColumn("source_channel",  F.lower(F.trim("source_channel")))
    .fillna({"source_channel": "unknown", "currency": "USD"})
    .filter(F.col("order_id").isNotNull())
    .withColumn("rn", F.row_number().over(dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("silver_processed_at", F.current_timestamp())
    .withColumn("data_zone", F.lit("silver"))
)

# Write as Delta Lake to Silver
(orders_silver.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .save(f"{SILVER}/orders_cleansed/"))

print(f"  Silver rows: {orders_silver.count()}")
print("  ✅ Orders Bronze → Silver complete")

## Step 2: Products Bronze → Silver

Reads raw product Parquet files, fixes data types,
removes duplicates, and writes clean Delta Lake table.

Key transformations:
- Cast product_id to Long
- Cast price to Decimal(18,2)
- Trim whitespace from category
- Deduplicate by product_id

In [0]:
# Cell 3 — Products: Bronze → Silver
print("\n[2/4] Processing products Bronze → Silver...")

products_raw = spark.read.parquet(f"{BRONZE}/products/{TODAY}/")

products_silver = (
    products_raw
    .withColumn("product_id", F.col("product_id").cast(LongType()))
    .withColumn("price",      F.col("price").cast(DecimalType(18,2)))
    .withColumn("category",   F.trim("category"))
    .dropDuplicates(["product_id"])
    .withColumn("silver_processed_at", F.current_timestamp())
)

(products_silver.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .save(f"{SILVER}/products_cleansed/"))

print(f"  Silver rows: {products_silver.count()}")
print("  ✅ Products Bronze → Silver complete")

## Step 3: Customers Bronze → Silver

Reads raw customer Parquet files from the Customer API.
Cleanses profiles including email, name formatting,
date parsing, and deduplication by customer_id.

Key transformations:
- Cast customer_id to Long
- Lowercase and trim email
- Parse signup_date and last_order_date to Date
- Cast total_spend, avg_order_value to Decimal
- Deduplicate by customer_id (keep latest ingested)

In [0]:
# Cell 4 — Customers: Bronze → Silver
print("\n[3/4] Processing customers Bronze → Silver...")

customers_raw = spark.read.parquet(f"{BRONZE}/customers/{TODAY}/")
print(f"  Raw rows: {customers_raw.count()}")

cust_window = Window.partitionBy("customer_id").orderBy(F.col("ingested_at").desc())

customers_silver = (
    customers_raw
    .withColumn("customer_id",          F.col("customer_id").cast(LongType()))
    .withColumn("email",                F.lower(F.trim("email")))
    .withColumn("full_name",            F.initcap(F.trim("full_name")))
    .withColumn("signup_date",          F.to_date("signup_date"))
    .withColumn("last_order_date",      F.to_date("last_order_date"))
    .withColumn("total_spend",          F.col("total_spend").cast(DecimalType(18,2)))
    .withColumn("avg_order_value",      F.col("avg_order_value").cast(DecimalType(18,2)))
    .withColumn("total_orders",         F.col("total_orders").cast(IntegerType()))
    .withColumn("days_since_last_order",F.col("days_since_last_order").cast(IntegerType()))
    .filter(F.col("customer_id").isNotNull())
    .withColumn("rn", F.row_number().over(cust_window))
    .filter(F.col("rn") == 1).drop("rn")
    .withColumn("silver_processed_at",  F.current_timestamp())
)

(customers_silver.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .save(f"{SILVER}/customers_cleansed/"))

print(f"  Silver rows: {customers_silver.count()}")
print("  ✅ Customers Bronze → Silver complete")

## Step 4: Customer Events Bronze → Silver

Reads raw customer behavioural events.
Events include: logins, product views, support tickets,
return requests, wishlist adds, newsletter unsubscribes.

These events feed the mart_customer_activity Gold table
which tracks customer engagement in the last 90 days.

Key transformations:
- Cast event_id, customer_id to Long
- Parse event_ts to proper timestamp
- Extract event_date and event_hour
- Lowercase event_type
- Deduplicate by event_id

In [0]:
# Cell 5 — Customer Events: Bronze → Silver
print("\n[4/4] Processing customer events Bronze → Silver...")

events_raw = spark.read.parquet(f"{BRONZE}/customer_events/{TODAY}/")
print(f"  Raw rows: {events_raw.count()}")

events_silver = (
    events_raw
    .withColumn("event_id",    F.col("event_id").cast(LongType()))
    .withColumn("customer_id", F.col("customer_id").cast(LongType()))
    .withColumn("event_ts",    F.to_timestamp("event_ts"))
    .withColumn("event_type",  F.lower(F.trim("event_type")))
    .withColumn("event_date",  F.to_date("event_ts"))
    .withColumn("event_hour",  F.hour("event_ts"))
    .dropDuplicates(["event_id"])
    .filter(F.col("customer_id").isNotNull())
    .withColumn("silver_processed_at", F.current_timestamp())
)

(events_silver.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .save(f"{SILVER}/customer_events_cleansed/"))

print(f"  Silver rows: {events_silver.count()}")
print("  ✅ Events Bronze → Silver complete")

In [0]:
# Cell 6 — Verification
print("\n=== Bronze → Silver Verification ===")

datasets = [
    ("orders_cleansed",          "orders"),
    ("products_cleansed",        "products"),
    ("customers_cleansed",       "customers"),
    ("customer_events_cleansed", "events"),
]

for silver_path, name in datasets:
    df = spark.read.format("delta").load(f"{SILVER}/{silver_path}/")
    print(f"  ✅ {name}: {df.count()} rows | columns: {len(df.columns)}")

print("\n=== All Silver tables verified ===")
dbutils.notebook.exit("SUCCESS")

## ✅ Bronze → Silver Complete

All 4 datasets successfully processed:
- orders_cleansed/ (Delta Lake)
- products_cleansed/ (Delta Lake)
- customers_cleansed/ (Delta Lake)
- customer_events_cleansed/ (Delta Lake)

Next: Run notebook 02_silver_to_gold to build Gold tables.